In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/citibike/default/layers/raw_files/*.csv")
)

print(f"Total Records: {df.count():,}")
print(f"Total Columns: {len(df.columns)}")

display(df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp, col

bronze_df = (
    df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
display(bronze_df.limit(5))

In [0]:
bronze_path = "/Volumes/citibike/default/layers/bronze/rides"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

In [0]:
# reack back bronze data
bronze_check = (
    spark.read
    .format("delta")
    .load("/Volumes/citibike/default/layers/bronze/rides")
)

print(f"Bronze Records: {bronze_check.count():,}")

display(bronze_check.limit(10))

In [0]:
%sql
drop table if exists citibike.default.bronze_rides

In [0]:
%sql
CREATE TABLE IF NOT EXISTS citibike.default.bronze_rides
USING DELTA
AS SELECT * FROM delta.`/Volumes/citibike/default/layers/bronze/rides`

In [0]:
%sql
use citibike.default

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM bronze_rides;

In [0]:
bronze_check.select(
    "member_casual",
    "rideable_type"
).distinct().display()